# Predict — ABSA Pipeline

Dự đoán aspect-based sentiment cho review tiếng Việt.

**Pipeline:** NER (ELECTRA) → aspect spans → SA (PhoBERT) per aspect + tổng thể.

## 1. Cấu hình

In [1]:
from os import path

ROOT         = "/Users/abc/Documents/Sentiment-analysis"
NER_MODEL    = path.join(ROOT, "models/NER")
SA_MODEL     = path.join(ROOT, "models/SA")
VNCORENLP_DIR = path.join(ROOT, "models/vncorenlp")
MAX_SEQ_LEN  = 256

## 2. VnCoreNLP

In [2]:
import os
import py_vncorenlp

os.makedirs(VNCORENLP_DIR, exist_ok=True)
py_vncorenlp.download_model(save_dir=VNCORENLP_DIR)
segmenter = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir=VNCORENLP_DIR)

VnCoreNLP model folder /Users/abc/Documents/Sentiment-analysis/models/vncorenlp already exists! Please load VnCoreNLP from this folder!
2026-03-05 16:30:07 INFO  WordSegmenter:24 - Loading Word Segmentation model


## 3. Load Models

In [3]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    AutoModelForSequenceClassification,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

ner_tokenizer = AutoTokenizer.from_pretrained(NER_MODEL)
ner_model     = AutoModelForTokenClassification.from_pretrained(NER_MODEL).to(device).eval()

sa_tokenizer  = AutoTokenizer.from_pretrained(SA_MODEL)
sa_model      = AutoModelForSequenceClassification.from_pretrained(SA_MODEL).to(device).eval()

print("NER labels:", list(ner_model.config.id2label.values()))
print("SA  labels:", list(sa_model.config.id2label.values()))

Device: cpu


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

NER labels: ['B-BATTERY', 'B-CAMERA', 'B-DESIGN', 'B-FEATURES', 'B-GENERAL', 'B-PERFORMANCE', 'B-PRICE', 'B-SCREEN', 'B-SER&ACC', 'B-STORAGE', 'I-BATTERY', 'I-CAMERA', 'I-DESIGN', 'I-FEATURES', 'I-GENERAL', 'I-PERFORMANCE', 'I-PRICE', 'I-SCREEN', 'I-SER&ACC', 'I-STORAGE', 'O']
SA  labels: ['NEGATIVE', 'NEUTRAL', 'POSITIVE']


## 4. Tiền xử lý

In [4]:
import unicodedata
import re

def clean_text(text):
    chars = [c if (unicodedata.category(c)[0] in ('L', 'N') or c == ' ') else ' ' for c in text]
    return re.sub(r' +', ' ', ''.join(chars)).strip().lower()


def preprocess_ner(text):
    """Chỉ clean — NER train không dùng word segmentation."""
    return clean_text(text)


def preprocess_sa(text):
    """Clean + word segment — SA train dùng VnCoreNLP."""
    segs = segmenter.word_segment(clean_text(text))
    return segs[0] if segs else clean_text(text)

## 5. Predict Functions

In [5]:
def predict_ner(text):
    """Trả về list[{"word": str, "tag": str}] với tag là nhãn aspect (bỏ prefix B-/I-)."""
    words  = text.split()
    inputs = ner_tokenizer(
        words,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LEN,
    ).to(device)

    with torch.no_grad():
        logits = ner_model(**inputs).logits

    pred_ids = torch.argmax(logits, dim=2)[0].tolist()
    word_ids = inputs.word_ids()

    # Lấy prediction của subword đầu tiên cho mỗi từ
    word_preds = {}
    for subword_idx, word_id in enumerate(word_ids):
        if word_id is not None and word_id not in word_preds:
            word_preds[word_id] = ner_model.config.id2label[pred_ids[subword_idx]]

    return [{"word": words[i], "tag": word_preds.get(i, "O")} for i in range(len(words))]


def predict_sa(text):
    """Trả về nhãn sentiment (NEGATIVE/NEUTRAL/POSITIVE)."""
    inputs = sa_tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_SEQ_LEN,
    ).to(device)

    with torch.no_grad():
        logits = sa_model(**inputs).logits

    pred_id = torch.argmax(logits, dim=1).item()
    return sa_model.config.id2label[pred_id]


def predict_absa(text):
    ner_text = preprocess_ner(text)
    sa_text  = preprocess_sa(text)

    # Bước 1: NER — tìm các aspect spans
    ner_result = predict_ner(ner_text)

    # Bước 2: Gom các từ liên tiếp cùng aspect thành span
    aspects = {}
    for item in ner_result:
        tag = item["tag"].split("-")[-1]
        if tag == "O":
            continue
        aspects.setdefault(tag, []).append(item["word"])

    if not aspects:
        return {"text": text, "labels": [], "sentiment": predict_sa(sa_text)}

    # Bước 3: SA cho từng aspect span
    labels = []
    for aspect, span_words in aspects.items():
        span     = " ".join(span_words)
        match    = re.search(re.escape(span), ner_text)
        if match is None:
            continue
        span_sa  = preprocess_sa(text[match.start():match.end()])
        pair     = f"{span_sa} : {sa_text}"
        sentiment = predict_sa(pair)
        labels.append([match.start(), match.end(), f"{aspect}#{sentiment}"])

    # Bước 4: SA tổng thể
    overall = predict_sa(sa_text)

    return {"text": text, "labels": labels, "sentiment": overall}

## 6. Demo

In [6]:
text = "Sp ổn, mỗi tội vân tay lúc nhận lúc không, nhân viên nhiệt tình, pin trâu, cả đêm tụt 1%"
result = predict_absa(text)

print("Text     :", result["text"])
print("Sentiment:", result["sentiment"])
print("Labels:")
for label in result["labels"]:
    aspect, sentiment = label[2].split("#")
    span = text[label[0]:label[1]]
    print(f"  [{aspect}] {span!r} → {sentiment}")

Text     : Sp ổn, mỗi tội vân tay lúc nhận lúc không, nhân viên nhiệt tình, pin trâu, cả đêm tụt 1%
Sentiment: POSITIVE
Labels:
  [GENERAL] 'Sp ổn' → POSITIVE
  [FEATURES] ' vân tay lúc nhận lúc khôn' → NEGATIVE
  [SER&ACC] ', nhân viên nhiệt tì' → POSITIVE
  [BATTERY] 'h, pin trâu, cả đêm t' → POSITIVE
